# Notebook C2 - Equity audit + Group-DRO mitigation + faithfulness across skin tones

Cached features from Notebook C1 (Fitzpatrick17k train/val/test + DDI external). CPU only -- no Kaggle needed.

Concept-bottleneck classifier: DermLIP zero-shot 7-point concept scores -> MLP -> **malignant**. We compare standard training (**ERM**) with **Group-DRO** (optimises the worst skin-tone group).

**What this notebook establishes -- with bootstrap CIs and significance tests, so we do not chase noise:**
- **Audit (headline):** DermLIP's malignancy detection carries a real, significant skin-tone AUROC gap (light > dark), quantified with a bootstrap CI on the gap **and** a permutation test.
- **Mitigation (Group-DRO):** an honest test of whether worst-group training closes the gap -- reported as a paired-bootstrap difference in worst-group AUROC. A CI spanning 0 is a negative result, not a fix.
- **H3 (open question):** whether explanation faithfulness (monotonicity of P(malignant) in the concepts) *itself* differs across skin tones -- reported with per-group bootstrap CIs so a small gap is not over-claimed.

**Input:** attach the Notebook C1 output (features_fitz.npz, features_ddi.npz).

In [ ]:
# --- 1. Load caches & decide train/eval setup ---
import os, glob, json
import numpy as np
import torch

GROUPS = ['light', 'mid', 'dark']
MIN_GROUP = 30        # a skin-tone group needs at least this many images to train on it
roots = ['/kaggle/input', '.', '..']

def find_one(name):
    for r in roots:
        hits = sorted(glob.glob(os.path.join(r, '**', name), recursive=True))
        if hits:
            return hits[0]
    return None

def load_tag(tag):
    fpz = find_one('features_' + tag + '.npz')
    if fpz is None:
        return None
    z = np.load(fpz, allow_pickle=True)
    C = z['concept_scores'].astype(np.float32)
    ok = np.isfinite(C).all(1) & np.isin(z['group'].astype('U8'), GROUPS)
    return {'C': torch.tensor(C[ok]), 'y': torch.tensor(z['malignant'][ok].astype(np.float32)),
            'mp': torch.tensor(z['malig_prob'][ok].astype(np.float32)),
            'g': z['group'].astype('U8')[ok], 'split': z['split'].astype('U8')[ok]}

def has_all_groups(d):
    return d is not None and all(int((d['g'] == g).sum()) >= MIN_GROUP for g in GROUPS)

def strat_split(g, y, frac_train=0.6, seed=0):
    rng = np.random.default_rng(seed)
    train = np.zeros(len(g), dtype=bool)
    yy = y.numpy()
    for grp in GROUPS:
        for lab in (0.0, 1.0):
            sel = np.where((g == grp) & (yy == lab))[0]
            rng.shuffle(sel)
            train[sel[:int(round(frac_train * len(sel)))]] = True
    return train

fitz, pad, ddi = load_tag('fitz'), load_tag('pad'), load_tag('ddi')

# Pick a diverse training set: Fitzpatrick17k > PAD (only if it has all tone groups) > DDI-internal split.
# PAD-UFES-20 is light-skin dominant (few 'dark'), so it usually falls through to DDI-internal.
if has_all_groups(fitz):
    SETUP, src = 'fitz', fitz
    tr, te = src['split'] == 'train', src['split'] == 'test'
elif has_all_groups(pad):
    SETUP, src = 'pad', pad
    tr, te = src['split'] == 'train', src['split'] == 'test'
else:
    assert has_all_groups(ddi), 'No cache has all 3 skin-tone groups with >= MIN_GROUP images. Run C1 with DDI (and ideally Fitzpatrick17k).'
    SETUP, src = 'ddi_internal', ddi
    tr = strat_split(ddi['g'], ddi['y'], 0.6, seed=0); te = ~tr

ADD_EXTERNAL = (SETUP != 'ddi_internal') and (ddi is not None)
EVAL_TAG = 'ddi' if SETUP == 'ddi_internal' else SETUP
Xtr, ytr, gtr = src['C'][tr], src['y'][tr], src['g'][tr]
Xte, yte, gte = src['C'][te], src['y'][te], src['g'][te]
print('SETUP:', SETUP, '| external DDI test:', ADD_EXTERNAL)
print('train groups:', {g: int((gtr == g).sum()) for g in GROUPS}, '| malig rate', round(float(ytr.mean()), 2))
print('test  groups:', {g: int((gte == g).sum()) for g in GROUPS}, '| malig rate', round(float(yte.mean()), 2))

In [ ]:
# --- 2. Metrics ---
def auc(y_true, score):
    y_true = np.asarray(y_true).astype(float); score = np.asarray(score).astype(float)
    pos = score[y_true == 1]; neg = score[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float('nan')
    allv = np.concatenate([pos, neg])
    rank = allv.argsort().argsort().astype(float) + 1
    return (rank[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))

def fwd(C, P):
    W1, b1, W2, b2 = P
    return (torch.relu(C @ W1 + b1) @ W2 + b2).squeeze(1)

def monotonicity(P, C):
    if len(C) == 0:
        return float('nan')
    ok = []
    for i in range(C.shape[1]):
        v1 = C.clone(); v1[:, i] = 1.0
        v0 = C.clone(); v0[:, i] = 0.0
        p1 = torch.sigmoid(fwd(v1, P)); p0 = torch.sigmoid(fwd(v0, P))
        ok.append((p1 >= p0 - 1e-6).float().mean().item())
    return float(np.mean(ok))

def eval_groups(P, C, y, g):
    prob = torch.sigmoid(fwd(C, P)).detach().numpy()
    res = {}
    for grp in GROUPS:
        m = (g == grp)
        if m.sum() > 0:
            res[grp] = {'n': int(m.sum()), 'auc': auc(y.numpy()[m], prob[m]), 'mono': monotonicity(P, C[m])}
    aucs = [res[k]['auc'] for k in res if not np.isnan(res[k]['auc'])]
    monos = [res[k]['mono'] for k in res if not np.isnan(res[k]['mono'])]
    res['_worst_auc'] = float(np.min(aucs)) if aucs else float('nan')
    res['_auc_gap'] = float(np.max(aucs) - np.min(aucs)) if aucs else float('nan')
    res['_mono_gap'] = float(np.max(monos) - np.min(monos)) if monos else float('nan')
    return res

def boot_ci(y, s, n=1000, seed=0):
    # AUROC with a percentile bootstrap 95% CI over test samples
    y = np.asarray(y).astype(float); s = np.asarray(s).astype(float)
    if len(y) == 0 or (y == 1).sum() == 0 or (y == 0).sum() == 0:
        return (float('nan'), float('nan'), float('nan'))
    rng = np.random.default_rng(seed); N = len(y); stats = []
    for _ in range(n):
        idx = rng.integers(0, N, N)
        a = auc(y[idx], s[idx])
        if not np.isnan(a):
            stats.append(a)
    if not stats:
        return (float(auc(y, s)), float('nan'), float('nan'))
    return (float(auc(y, s)), float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5)))


In [ ]:
# --- 2c. Rigor helpers: bootstrap CIs on faithfulness + significance tests on the tone gap ---
def mono_per_sample(P, C):
    """Per-sample monotonicity in [0,1]: fraction of the 7 concepts whose forced
    presence does not lower P(malignant). Mean over samples == monotonicity()."""
    if len(C) == 0:
        return np.zeros(0, dtype=float)
    inds = []
    for i in range(C.shape[1]):
        v1 = C.clone(); v1[:, i] = 1.0
        v0 = C.clone(); v0[:, i] = 0.0
        p1 = torch.sigmoid(fwd(v1, P)); p0 = torch.sigmoid(fwd(v0, P))
        inds.append((p1 >= p0 - 1e-6).float().detach().numpy())
    return np.mean(inds, axis=0)

def mono_per_sample_ens(heads, C):
    if len(C) == 0:
        return np.zeros(0, dtype=float)
    return np.mean([mono_per_sample(P, C) for P in heads], axis=0)

def mean_ci(vals, n=1000, seed=0):
    """Percentile bootstrap 95% CI for the mean of a per-sample score."""
    vals = np.asarray(vals, dtype=float)
    if len(vals) == 0:
        return (float('nan'), float('nan'), float('nan'))
    rng = np.random.default_rng(seed); N = len(vals); st = []
    for _ in range(n):
        st.append(vals[rng.integers(0, N, N)].mean())
    return (float(vals.mean()), float(np.percentile(st, 2.5)), float(np.percentile(st, 97.5)))

def gap_ci(y, s, g, a='light', b='dark', n=1000, seed=0):
    """Bootstrap 95% CI for AUROC(a) - AUROC(b), resampling within each group."""
    y = np.asarray(y).astype(float); s = np.asarray(s).astype(float)
    ya, sa = y[g == a], s[g == a]; yb, sb = y[g == b], s[g == b]
    obs = auc(ya, sa) - auc(yb, sb)
    rng = np.random.default_rng(seed); st = []
    for _ in range(n):
        ia = rng.integers(0, len(ya), len(ya)); ib = rng.integers(0, len(yb), len(yb))
        dd = auc(ya[ia], sa[ia]) - auc(yb[ib], sb[ib])
        if not np.isnan(dd):
            st.append(dd)
    lo, hi = (float(np.percentile(st, 2.5)), float(np.percentile(st, 97.5))) if st else (float('nan'), float('nan'))
    return (float(obs), lo, hi)

def gap_perm_test(y, s, g, a='light', b='dark', n=5000, seed=0):
    """Permutation test for AUROC(a) > AUROC(b): shuffle group labels among the
    pooled a+b samples and rebuild the gap null distribution."""
    y = np.asarray(y).astype(float); s = np.asarray(s).astype(float)
    ya, sa = y[g == a], s[g == a]; yb, sb = y[g == b], s[g == b]
    obs = auc(ya, sa) - auc(yb, sb)
    yy = np.concatenate([ya, yb]); ss = np.concatenate([sa, sb])
    na = len(ya); N = len(yy); rng = np.random.default_rng(seed); perm = []
    for _ in range(n):
        idx = rng.permutation(N)
        perm.append(auc(yy[idx[:na]], ss[idx[:na]]) - auc(yy[idx[na:]], ss[idx[na:]]))
    perm = np.asarray(perm)
    return {'gap': float(obs), 'p_one_sided': float((perm >= obs).mean()),
            'p_two_sided': float((np.abs(perm) >= abs(obs)).mean())}

def worst_diff_ci(y, g, ens_erm, ens_gdro, n=1000, seed=0):
    """Paired bootstrap 95% CI for (worst-group AUROC gdro) - (worst-group AUROC erm).
    A CI containing 0 => Group-DRO gives no significant worst-group gain."""
    y = np.asarray(y).astype(float)
    idx_by_g = {grp: np.where(g == grp)[0] for grp in GROUPS if (g == grp).sum() > 0}
    def worst(prob, picks):
        a = [auc(y[ix], prob[ix]) for ix in picks]
        a = [v for v in a if not np.isnan(v)]
        return min(a) if a else float('nan')
    full = list(idx_by_g.values())
    obs = worst(ens_gdro, full) - worst(ens_erm, full)
    rng = np.random.default_rng(seed); st = []
    for _ in range(n):
        picks = [ix[rng.integers(0, len(ix), len(ix))] for ix in idx_by_g.values()]
        dd = worst(ens_gdro, picks) - worst(ens_erm, picks)
        if not np.isnan(dd):
            st.append(dd)
    lo, hi = (float(np.percentile(st, 2.5)), float(np.percentile(st, 97.5))) if st else (float('nan'), float('nan'))
    return (float(obs), lo, hi)

In [ ]:
# --- 2b. Zero-shot equity audit: DermLIP P(malignant) per skin tone, bootstrap 95% CIs ---
# This is the headline audit (no training) -- non-overlapping CIs => a real skin-tone disparity.
zs_audit = {}
zs_sets = {EVAL_TAG + '_test': (yte, gte, src['mp'][te])}
if ADD_EXTERNAL:
    zs_sets['ddi_external'] = (ddi['y'], ddi['g'], ddi['mp'])
for name, (y, g, mp) in zs_sets.items():
    yv = y.numpy(); mpv = mp.numpy()
    zs_audit[name] = {}
    print('== zero-shot malignancy AUROC [' + name + '] ==')
    for grp in GROUPS:
        m = (g == grp)
        mean, lo, hi = boot_ci(yv[m], mpv[m])
        zs_audit[name][grp] = {'n': int(m.sum()), 'auc': mean, 'ci_lo': lo, 'ci_hi': hi}
        print('  {0:5s} n={1:4d} auc={2:.3f}  95% CI [{3:.3f}, {4:.3f}]'.format(grp, int(m.sum()), mean, lo, hi))


In [ ]:
# --- 3. Training: ERM vs Group-DRO ---
HID = 16

def init(seed):
    g = torch.Generator().manual_seed(seed)
    W1 = (torch.randn(Xtr.shape[1], HID, generator=g) * 0.3).requires_grad_(True)
    b1 = torch.zeros(HID, requires_grad=True)
    W2 = (torch.randn(HID, 1, generator=g) * 0.3).requires_grad_(True)
    b2 = torch.zeros(1, requires_grad=True)
    return [W1, b1, W2, b2]

def pos_weight():
    npos = float((ytr == 1).sum()); nneg = float((ytr == 0).sum())
    return torch.tensor(max(nneg, 1.0) / max(npos, 1.0))

def train_head(mode, seed=0, iters=1500, lr=0.05, eta=1.0):
    P = init(seed)
    opt = torch.optim.Adam(P, lr=lr)
    bce = torch.nn.functional.binary_cross_entropy_with_logits
    pw = pos_weight()
    q = torch.ones(len(GROUPS)) / len(GROUPS)
    gmask = [torch.tensor(gtr == grp) for grp in GROUPS]
    for it in range(iters):
        logit = fwd(Xtr, P)
        per = bce(logit, ytr, pos_weight=pw, reduction='none')
        if mode == 'erm':
            loss = per.mean()
        else:
            lg = torch.stack([per[m].mean() if m.any() else torch.tensor(0.0) for m in gmask])
            q = q * torch.exp(eta * lg.detach()); q = q / q.sum()
            loss = (q * lg).sum()
        opt.zero_grad(); loss.backward(); opt.step()
    return P

In [ ]:
# --- 4. Run (3 seeds) and aggregate, with per-group bootstrap CIs on AUROC and faithfulness ---
SEEDS = [0, 1, 2]
ARTIFACTS = {}   # es -> mode -> {heads, ens, y, g} kept in memory for downstream CIs / tests

def run(mode, data, es):
    C, y, g = data
    keys = ['_worst_auc', '_auc_gap', '_mono_gap']
    agg = {k: [] for k in keys}
    probs = []; heads = []
    for s in SEEDS:
        P = train_head(mode, seed=s)
        heads.append(P)
        r = eval_groups(P, C, y, g)
        for k in keys:
            agg[k].append(r[k])
        probs.append(torch.sigmoid(fwd(C, P)).detach().numpy())
    ens = np.mean(probs, axis=0)  # ensemble P(malignant) across seeds
    yv = y.numpy()
    ARTIFACTS.setdefault(es, {})[mode] = {'heads': heads, 'ens': ens, 'y': yv, 'g': g}
    out = {k: float(np.nanmean(agg[k])) for k in keys}
    pg = {}
    for grp in GROUPS:
        m = (g == grp)
        amean, alo, ahi = boot_ci(yv[m], ens[m])
        mps = mono_per_sample_ens(heads, C[m])       # seed-ensembled per-sample faithfulness
        mmean, mlo, mhi = mean_ci(mps)               # bootstrap 95% CI over samples
        pg[grp] = {'auc': amean, 'ci_lo': alo, 'ci_hi': ahi,
                   'mono': mmean, 'mono_lo': mlo, 'mono_hi': mhi}
    out['per_group'] = pg
    return out

evalsets = {EVAL_TAG + '_test': (Xte, yte, gte)}
if ADD_EXTERNAL:
    evalsets['ddi_external'] = (ddi['C'], ddi['y'], ddi['g'])

results = {}
for es, data in evalsets.items():
    results[es] = {'erm': run('erm', data, es), 'gdro': run('gdro', data, es)}
    print('==', es, '==')
    for mode in ['erm', 'gdro']:
        r = results[es][mode]
        pg = ' '.join('{0}:auc={1:.3f}[{2:.3f},{3:.3f}]'.format(g, r['per_group'][g]['auc'], r['per_group'][g]['ci_lo'], r['per_group'][g]['ci_hi']) for g in GROUPS)
        print('  {0:5s} worst_auc={1:.3f} auc_gap={2:.3f} mono_gap={3:.3f}'.format(mode, r['_worst_auc'], r['_auc_gap'], r['_mono_gap']))
        print('        ', pg)

In [ ]:
# --- 4b. Significance: is the tone gap real, and does Group-DRO close it? ---
for es in results:
    A = ARTIFACTS[es]
    y, g = A['erm']['y'], A['erm']['g']
    # (i) prediction disparity: light-vs-dark AUROC gap (ERM ensemble) -- CI + permutation p
    gobs, glo, ghi = gap_ci(y, A['erm']['ens'], g, 'light', 'dark')
    perm = gap_perm_test(y, A['erm']['ens'], g, 'light', 'dark')
    # (ii) Group-DRO mitigation: paired-bootstrap diff in worst-group AUROC
    wobs, wlo, whi = worst_diff_ci(y, g, A['erm']['ens'], A['gdro']['ens'])
    pe = results[es]['erm']['per_group']
    results[es]['stats'] = {
        'auc_gap_light_minus_dark': {'erm_gap': gobs, 'ci_lo': glo, 'ci_hi': ghi,
                                     'perm_p_one_sided': perm['p_one_sided'], 'perm_p_two_sided': perm['p_two_sided']},
        'gdro_minus_erm_worst_auc': {'diff': wobs, 'ci_lo': wlo, 'ci_hi': whi},
        'mono_light_vs_dark': {'light': [pe['light']['mono'], pe['light']['mono_lo'], pe['light']['mono_hi']],
                               'dark': [pe['dark']['mono'], pe['dark']['mono_lo'], pe['dark']['mono_hi']]}}
    print('== significance [' + es + '] ==')
    print('  AUROC gap light-dark (ERM) = {0:.3f}  95% CI [{1:.3f}, {2:.3f}]  perm p(1-sided)={3:.4f}'.format(gobs, glo, ghi, perm['p_one_sided']))
    print('  Group-DRO - ERM worst-group AUROC = {0:+.3f}  95% CI [{1:.3f}, {2:.3f}]  {3}'.format(
        wobs, wlo, whi, 'significant' if (wlo > 0 or whi < 0) else 'n.s. (CI spans 0)'))
    print('  faithfulness (mono): light={0:.3f}[{1:.3f},{2:.3f}]  dark={3:.3f}[{4:.3f},{5:.3f}]'.format(
        pe['light']['mono'], pe['light']['mono_lo'], pe['light']['mono_hi'],
        pe['dark']['mono'], pe['dark']['mono_lo'], pe['dark']['mono_hi']))

In [ ]:
# --- 5. Verdict (honest, CI-gated) + save ---
for es in results:
    e = results[es]['erm']; d = results[es]['gdro']; st = results[es].get('stats', {})
    gp = st.get('auc_gap_light_minus_dark', {}); wd = st.get('gdro_minus_erm_worst_auc', {})
    pe = e['per_group']
    print('[' + es + ']')
    print('  AUDIT  light-vs-dark AUROC gap (ERM) = {0:.3f}  95% CI [{1:.3f}, {2:.3f}]  perm p={3:.4f}'.format(
        gp.get('erm_gap', float('nan')), gp.get('ci_lo', float('nan')), gp.get('ci_hi', float('nan')), gp.get('perm_p_one_sided', float('nan'))))
    print('         -> disparity {0}'.format('SIGNIFICANT (gap CI excludes 0)' if gp.get('ci_lo', -1) > 0 else 'not significant (gap CI spans 0)'))
    print('  H2     Group-DRO - ERM worst-group AUROC = {0:+.3f}  95% CI [{1:.3f}, {2:.3f}]'.format(
        wd.get('diff', float('nan')), wd.get('ci_lo', float('nan')), wd.get('ci_hi', float('nan'))))
    print('         -> {0}'.format('Group-DRO helps (CI excludes 0)' if wd.get('ci_lo', -1) > 0 else 'NEGATIVE: no significant gain (CI spans 0)'))
    print('  H3     faithfulness light={0:.3f}[{1:.3f},{2:.3f}] vs dark={3:.3f}[{4:.3f},{5:.3f}]  (mono_gap={6:.3f})'.format(
        pe['light']['mono'], pe['light']['mono_lo'], pe['light']['mono_hi'],
        pe['dark']['mono'], pe['dark']['mono_lo'], pe['dark']['mono_hi'], e['_mono_gap']))
    overlap = not (pe['light']['mono_lo'] > pe['dark']['mono_hi'] or pe['dark']['mono_lo'] > pe['light']['mono_hi'])
    print('         -> {0}'.format('INCONCLUSIVE: light/dark faithfulness CIs overlap (open question)' if overlap else 'faithfulness gap significant (CIs disjoint)'))

OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
out_obj = {'setup': SETUP, 'zero_shot_audit': zs_audit, 'concept_bottleneck': results}
fname = 'nbC2_' + SETUP + '_results.json'
with open(os.path.join(OUT, fname), 'w') as fh:
    json.dump(out_obj, fh, indent=2)
print('saved', os.path.join(OUT, fname))

## How to read this

- **AUDIT (headline):** the light-vs-dark AUROC gap with its bootstrap 95% CI and a permutation p-value. If the gap CI excludes 0 (and p is small), the skin-tone disparity is statistically real -- this is the paper's anchor result.
- **H2 / mitigation:** Group-DRO minus ERM worst-group AUROC, as a **paired** bootstrap 95% CI. If that CI spans 0, Group-DRO gives **no significant gain** -- an honest negative result, reported as such, not a fix.
- **H3 / faithfulness (open question):** per-group monotonicity with bootstrap CIs. Overlapping light/dark CIs => the faithfulness gap is within noise; we report it as a suggestive trend and future work, **not** a finding.

Caveats: concepts are DermLIP zero-shot (no concept ground truth on these datasets), so monotonicity (~0.6, near the 0.5 floor) reflects imperfect concepts on clinical photos -- that weak substrate is exactly why H3 cannot be cleanly resolved here, and why closing it needs concept supervision on diverse clinical images (future work). Fitzpatrick17k tone labels are image-estimated; DDI (biopsy-proven) is the more reliable signal. Cross-dataset transfer (train Fitz -> test DDI) collapses toward chance -- a domain-shift result worth reporting, not hiding.